# 05 Limitations: Missing Data And Sample Reliability

This notebook documents the missing-data limitation after the systematic workflow in notebook 01. The workflow now distinguishes between observed-only key variables and within-country-imputed eligible controls, so the main question is no longer just whether data is missing, but where missingness still constrains estimation after applying the country-panel handling rules.


## Table Of Contents

- [01 Setup](#01-Setup)
- [02 Load Missingness And Model Artifacts](#02-Load-Missingness-And-Model-Artifacts)
- [03 Missing-Data Handling Rules](#03-Missing-Data-Handling-Rules)
- [04 Missingness Mechanism Diagnostics](#04-Missingness-Mechanism-Diagnostics)
- [05 Control Imputation Audit](#05-Control-Imputation-Audit)
- [06 Sample-Loss Implications](#06-Sample-Loss-Implications)


## 01 Setup

Load paths and dependencies for the limitations and missing-data discussion.


In [9]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import OUTPUTS_DIR, PROCESSED_PANEL_FILE

PROCESSED_FILE = PROCESSED_PANEL_FILE

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')

## 02 Load Missingness And Model Artifacts

Read the clean panel, missingness diagnostics, imputation log, and model sample outputs.


In [10]:
clean_panel = pd.read_csv(PROCESSED_FILE)
model_status = pd.read_csv(OUTPUTS_DIR / 'model_status.csv')
model_sample_summary = pd.read_csv(OUTPUTS_DIR / 'model_sample_summary.csv')
model_sample_audit = pd.read_csv(OUTPUTS_DIR / 'model_sample_audit.csv')
coverage_by_country = pd.read_csv(OUTPUTS_DIR / 'coverage_by_country.csv')
missingness_mechanism = pd.read_csv(OUTPUTS_DIR / 'missingness_mechanism_diagnostics.csv')
missingness_handling = pd.read_csv(OUTPUTS_DIR / 'missingness_handling_summary.csv')
control_imputation_log = pd.read_csv(OUTPUTS_DIR / 'control_imputation_log.csv')

clean_panel.head()

,country_id,country_code,country,year,fdi_pct_gdp,fdi_pct_gdp_winsorized,broad_money_pct_gdp,trade_pct_gdp,inflation_gdp_deflator_pct,deposit_interest_rate_pct,real_interest_rate_pct,lending_interest_rate_pct,hc_human_capital_index,ln_gdppc,xr_dep_pct,xr_dep_pct_winsorized,ln_population_total,ln_tourism_arrivals
0,1,BRN,Brunei Darussalam,2010,3.5071,3.5071,67.2720,95.3715,4.9801,0.4705,0.4952,5.5000,2.6970,10.4613,NaN,NaN,12.8799,15.2641
1,1,BRN,Brunei Darussalam,2011,3.7311,3.7311,59.3805,99.5379,20.1808,0.3957,-12.2156,5.5000,2.7215,10.7447,-8.0608,-8.0608,12.8977,15.2641
2,1,BRN,Brunei Darussalam,2012,4.5406,4.5406,58.6565,105.6409,1.2203,0.2315,4.2282,5.5000,2.7325,10.7572,-0.6657,-0.6657,12.9130,15.2641
3,1,BRN,Brunei Darussalam,2013,4.2867,4.2867,62.5754,110.9396,-2.8235,0.2843,8.5653,5.5000,2.7436,10.6920,0.1279,0.1279,12.9268,15.2641
4,1,BRN,Brunei Darussalam,2014,3.3566,3.3566,67.4986,102.4210,-1.8460,0.3000,7.4842,5.5000,2.7549,10.6220,1.2608,1.2608,12.9402,15.2641


## 03 Missing-Data Handling Rules

Start with the preprocessing policy. This tells the reader which variables were kept observed-only and which controls were cautiously imputed.

Document which variables were filled, kept observed-only, or treated as limitations.


In [11]:
missingness_handling[['variable', 'role_group', 'severity_band', 'structural_missing_countries', 'mechanism_assessment', 'recommended_handling']]


,variable,role_group,severity_band,structural_missing_countries,mechanism_assessment,recommended_handling
0,hc_human_capital_index,control,<10%,1,MCAR_not_rejected,leave_missing_and_report_limitation
1,inflation_gdp_deflator_pct,control,<10%,0,complete,within_country_interpolate_then_edge_fill
2,ln_gdppc,control,<10%,0,complete,within_country_interpolate_then_edge_fill
3,ln_population_total,control,<10%,0,complete,within_country_interpolate_then_edge_fill
4,ln_tourism_arrivals,control,10-30%,0,MCAR_not_rejected,within_country_interpolate_then_edge_fill_keep...
5,trade_pct_gdp,control,10-30%,1,MCAR_not_rejected,within_country_interpolate_then_edge_fill_keep...
6,xr_dep_pct,control,<10%,0,MCAR_not_rejected,within_country_interpolate_and_edge_fill_excep...
7,fdi_pct_gdp,dependent,<10%,0,complete,observed_only_no_imputation
8,broad_money_pct_gdp,key_independent,10-30%,0,MCAR_not_rejected,observed_only_no_imputation
9,deposit_interest_rate_pct,key_independent,10-30%,0,MCAR_not_rejected,observed_only_no_imputation


## 04 Missingness Mechanism Diagnostics

Use these diagnostics to justify a conservative MAR-or-worse framing rather than claiming missingness is harmless.

Use mechanism diagnostics to justify a conservative MAR-or-worse framing.


In [12]:
missingness_mechanism[['variable', 'missing_rate', 'model_status', 'significant_predictors', 'mechanism_assessment']]


,variable,missing_rate,model_status,significant_predictors,mechanism_assessment
0,fdi_pct_gdp,0.0000,not_needed,NaN,complete
1,broad_money_pct_gdp,0.1364,fit,NaN,MCAR_not_rejected
2,trade_pct_gdp,0.1364,fit,NaN,MCAR_not_rejected
3,inflation_gdp_deflator_pct,0.0000,not_needed,NaN,complete
4,deposit_interest_rate_pct,0.1429,fit,NaN,MCAR_not_rejected
5,real_interest_rate_pct,0.2403,fit,NaN,MCAR_not_rejected
6,lending_interest_rate_pct,0.2403,fit,NaN,MCAR_not_rejected
7,hc_human_capital_index,0.0909,fit,NaN,MCAR_not_rejected
8,ln_gdppc,0.0000,not_needed,NaN,complete
9,xr_dep_pct,0.0909,fit,NaN,MCAR_not_rejected


## 05 Control Imputation Audit

Show the exact scope of imputation. Core FDI and monetary-policy variables should remain observed-only.

Show what the preprocessing notebook actually filled and what remained missing.


In [13]:
control_imputation_log


,variable,handling_applied,missing_before,missing_after,filled_values
0,trade_pct_gdp,within_country_interpolate_then_edge_fill_keep...,21,14,7
1,inflation_gdp_deflator_pct,within_country_interpolate_then_edge_fill,0,0,0
2,ln_gdppc,within_country_interpolate_then_edge_fill,0,0,0
3,xr_dep_pct,within_country_interpolate_and_edge_fill_excep...,14,11,3
4,ln_population_total,within_country_interpolate_then_edge_fill,0,0,0
5,ln_tourism_arrivals,within_country_interpolate_then_edge_fill_keep...,41,0,41
6,hc_human_capital_index,leave_missing_and_report_limitation,14,14,0


## 06 Model Sample Limits

Summarize how complete-case estimation changes rows, countries, and years across models.

Summarize how observed-only key variables constrain each complete-case model.


In [14]:
model_sample_summary.merge(
    model_status[['model_id', 'workbook_model']],
    on='model_id',
    how='left',
)[['model_id', 'workbook_model', 'rows_used', 'rows_dropped', 'countries_used', 'years_used']].sort_values('model_id').reset_index(drop=True)


,model_id,workbook_model,rows_used,rows_dropped,countries_used,years_used
0,M1_baseline_liquidity,M1 - Baseline liquidity,112,42,9,13
1,M2_main_monetary_policy,M2 - Deposit-rate channel,96,58,8,13
2,M3_lagged_main_model,M3 - Lagged deposit-rate robustness,91,63,8,12
3,M4_real_interest_robustness,M4 - Real-rate channel,96,58,8,13
4,M5_lending_rate_robustness,M5 - Lending rate robustness,96,58,8,13
5,M6a_tourism_robustness_from_M2,M6a - Tourism robustness from M2,96,58,8,13
6,M6b_tourism_robustness_from_M4,M6b - Tourism robustness from M4,96,58,8,13
7,M7a_human_capital_robustness_from_M2,M7a - Human capital sensitivity from M2,83,71,7,13
8,M7b_human_capital_robustness_from_M4,M7b - Human capital sensitivity from M4,83,71,7,13


## 07 Country Coverage Constraints

Inspect country-level coverage for sparse controls and robustness models.

Inspect country-level coverage for the sparse controls and robustness models.


In [15]:
coverage_by_country[
    [
        'country',
        'deposit_interest_rate_pct',
        'real_interest_rate_pct',
        'lending_interest_rate_pct',
        'broad_money_pct_gdp',
        'trade_pct_gdp',
        'xr_dep_pct',
        'ln_tourism_arrivals',
        'hc_human_capital_index',
    ]
].sort_values('country').reset_index(drop=True)


,country,deposit_interest_rate_pct,real_interest_rate_pct,lending_interest_rate_pct,broad_money_pct_gdp,trade_pct_gdp,xr_dep_pct,ln_tourism_arrivals,hc_human_capital_index
0,Brunei Darussalam,14,14,14,14,14,13,14,14
1,Cambodia,14,0,0,14,14,13,14,14
2,Indonesia,14,14,14,14,14,13,14,14
3,Lao PDR,1,1,1,1,14,13,14,14
4,Malaysia,14,14,14,14,14,13,14,14
5,Myanmar,11,11,11,11,0,13,14,14
6,Philippines,10,10,10,13,14,13,14,14
7,Singapore,12,12,12,11,14,13,14,14
8,Thailand,14,14,14,14,14,13,14,14
9,Timor-Leste,14,13,13,14,14,13,14,0


## 08 Adjacent Model Sample Losses

Identify which countries and variables drive sample changes between adjacent models.

Identify which countries and variables drive sample changes between adjacent models.


In [16]:
model_sample_audit[['comparison_id', 'base_rows', 'added_rows', 'rows_lost_from_base', 'lost_countries', 'top_loss_drivers']]


,comparison_id,base_rows,added_rows,rows_lost_from_base,lost_countries,top_loss_drivers
0,M1_to_M2,112,96,16,"Cambodia, Philippines",deposit_interest_rate_pct: 3
1,M2_to_M3_lagged_proxy,96,91,8,"Brunei Darussalam, Indonesia, Malaysia, Philip...",xr_dep_pct_lag1: 8
2,M2_to_M5_lending_robustness,96,96,0,NaN,NaN
3,M2_to_M7a_appendix_hc,96,83,13,Timor-Leste,hc_human_capital_index: 13
4,M4_to_M7b_appendix_hc,96,83,13,Timor-Leste,hc_human_capital_index: 13


## 09 Thesis-Facing Interpretation

State the thesis-facing caveats: main inference comes from M2/M3; sparse-control models are sensitivity checks.


The current workflow implies the following interpretation:

- The dependent variable and key monetary-policy variables are kept on observed data only.
- Eligible controls are handled with within-country interpolation and edge filling where the workflow allows it.
- Structural gaps are still left missing rather than fabricated.
- Therefore, any remaining sample loss in the main and robustness regressions should be interpreted as substantive data sparsity, not as a failure to apply a missingness policy.
